# บทที่ 6 — เทียบ 4 สถาปัตยกรรม

<sub>บทเรียนที่ 6 จาก 8 &nbsp;·&nbsp; [← บทที่ 5](05_first_model_lstm.ipynb) · [สารบัญ](README.md) · [บทที่ 7 →](07_reading_results_critically.ipynb)</sub>

## เป้าหมายของบทนี้

เมื่อจบบทนี้คุณจะ:

- รู้ว่า CNN-LSTM, BiLSTM, Transformer ต่างจาก LSTM ธรรมดาอย่างไร
- เข้าใจว่าการเปรียบเทียบที่ยุติธรรมต้องคุมตัวแปรอะไรบ้าง
- เทรนทั้ง 4 แบบแล้วเทียบผลด้วยตัวเอง
- ตอบได้ว่าโมเดลใหญ่กว่าดีกว่าเสมอไหม

> **บทนี้ไม่ต้องใช้ข้อมูลดิบ** — ใช้ไฟล์ `outputs/features_cache.npz` ที่อยู่ใน repo อยู่แล้ว รันได้เลย

---

## 6.1 อีก 3 วิธีในการอ่านลำดับเวลา

บทที่แล้วเราใช้ LSTM ซึ่งอ่านข้อมูลทีละจุดจากอดีตไปปัจจุบัน
ยังมีวิธีอื่นอีก มาดูว่าแต่ละแบบคิดต่างกันอย่างไร

### BiLSTM — อ่านสองทิศทาง

LSTM ธรรมดาอ่านจากซ้ายไปขวา จุดที่ 1 จึงไม่เคยรู้ว่าจุดที่ 20 เป็นอย่างไร

**BiLSTM** อ่านสองรอบ — ไปข้างหน้าและย้อนกลับ แล้วรวมผลกัน ทำให้ทุกจุดเห็นบริบททั้งสองด้าน

> ⚠️ ข้อควรระวัง: มันใช้ข้อมูล "อนาคต" ภายในหน้าต่าง จึงเหมาะกับการวิเคราะห์ย้อนหลัง
> ไม่ใช่การทำนายแบบ real-time

### CNN-LSTM — สรุปก่อน แล้วค่อยอ่าน

**CNN (Convolutional Neural Network)** เก่งเรื่องจับ "รูปแบบเฉพาะที่" —
เลื่อนหน้าต่างเล็ก ๆ ไปตามข้อมูลแล้วมองหาลวดลายที่คุ้นเคย

แนวคิดคือให้ CNN สรุปรูปแบบในช่วงสั้น ๆ ก่อน แล้วส่งผลสรุปให้ LSTM อ่านต่อ

> ⚠️ แต่มี MaxPool 2 ชั้นที่บีบลำดับจาก 20 เหลือ 5 จุด — จำจุดนี้ไว้ เดี๋ยวจะได้เห็นผล

### Transformer — มองเห็นกันหมดพร้อมกัน

แทนที่จะไล่ทีละจุด **self-attention** ให้ทุกจุดเวลามองเห็นกันได้ทั้งหมดในคราวเดียว
แล้วโมเดลเรียนรู้เองว่าควรให้น้ำหนักจุดไหน

เพราะไม่ได้ไล่ตามลำดับ จึงต้องใส่ **Positional Encoding** เพื่อบอกว่าจุดไหนมาก่อนหลัง

In [ ]:
# ── ตั้งค่าให้ notebook มองเห็นโค้ดใน src/ ──
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# ── หาฟอนต์ที่แสดงภาษาไทยได้ ไม่งั้นข้อความในกราฟจะกลายเป็นสี่เหลี่ยม ──
_installed = {f.name for f in fm.fontManager.ttflist}
for _f in ["Noto Sans Thai", "Leelawadee UI", "Tahoma", "TH Sarabun New", "Angsana New"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        plt.rcParams["axes.unicode_minus"] = False    # ฟอนต์ไทยมักไม่มีเครื่องหมายลบแบบ unicode
        print("ฟอนต์กราฟ:", _f)
        break
else:
    print("[หมายเหตุ] ไม่พบฟอนต์ไทย - ข้อความไทยในกราฟอาจแสดงเป็นสี่เหลี่ยม")
    print("           Windows/macOS มักมีอยู่แล้ว ส่วน Linux ลง: sudo apt install fonts-thai-tlwg")

print("project root:", ROOT)

In [ ]:
import torch
import torch.nn as nn
import time, copy
from scipy.stats import pearsonr

from src.paths import FEATURES_CACHE
from src.data_loader import compute_health_index
from src.dataset import split_dataset
from src.model import CNNLSTM
from src.models_extra import PureLSTM, BiLSTM, TransformerModel

device = "cuda" if torch.cuda.is_available() else "cpu"

features = np.load(FEATURES_CACHE)["features"]
labels = compute_health_index(features, window=7)

WINDOW = 20
train_loader, val_loader, test_loader, _ = split_dataset(
    features, labels, window_size=WINDOW, batch_size=32,
    shuffle_split=True, random_seed=42,
)
print("device:", device)

## 6.2 กติกาการเปรียบเทียบที่ยุติธรรม

ถ้าเทียบโมเดลแล้วให้เงื่อนไขต่างกัน ผลที่ได้จะไม่มีความหมาย
สิ่งที่ต้องคุมให้เหมือนกันทุกโมเดล:

| ตัวแปร | ค่า |
|---|---|
| ข้อมูล | split เดียวกัน (seed 42) |
| คำตอบ | Health Index เดียวกัน |
| ขนาดข้อมูลเข้า | (batch, 20, 56) |
| Loss | MSE |
| Optimizer | AdamW, lr 5e-4, weight decay 1e-4 |
| Gradient clipping | 1.0 |
| Early stopping | patience 15 |
| Seed | รีเซ็ตก่อนสร้างทุกโมเดล |

**สิ่งเดียวที่ต่างกันคือสถาปัตยกรรม**

ข้อสุดท้ายสำคัญมาก — ถ้าไม่รีเซ็ต seed โมเดลแต่ละตัวจะเริ่มจากน้ำหนักสุ่มคนละชุด
ความต่างที่เห็นอาจมาจากโชคของการสุ่ม ไม่ใช่สถาปัตยกรรม

In [ ]:
def train_model(model, epochs=60, patience=15, lr=5e-4, seed=42):
    """เทรนโมเดลด้วยเงื่อนไขมาตรฐานเดียวกันทุกตัว"""
    torch.manual_seed(seed)
    model = model.to(device)

    loss_fn = nn.MSELoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    hist = {"train": [], "val": []}
    best_val, best_state, wait = float("inf"), None, 0

    t0 = time.time()
    for epoch in range(epochs):
        # --- train ---
        model.train()
        total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(xb)
        hist["train"].append(total / len(train_loader.dataset))

        # --- validate ---
        model.eval()
        total = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                total += loss_fn(model(xb), yb).item() * len(xb)
        va = total / len(val_loader.dataset)
        hist["val"].append(va)

        if va < best_val:
            best_val, best_state, wait = va, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)

    # --- test ---
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(device)).cpu().numpy())
            targets.append(yb.numpy())
    preds, targets = np.concatenate(preds), np.concatenate(targets)

    return {
        "rmse": float(np.sqrt(np.mean((preds - targets) ** 2))),
        "mae": float(np.mean(np.abs(preds - targets))),
        "r": float(pearsonr(targets, preds)[0]),
        "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "epochs": len(hist["train"]),
        "seconds": time.time() - t0,
        "history": hist,
        "preds": preds,
        "targets": targets,
    }


print("พร้อมเทรน")

## 6.3 เทรนทั้ง 4 แบบ

ใช้เวลารวมประมาณ 1 นาทีบน CPU

In [ ]:
MODELS = {
    "CNN-LSTM":    lambda: CNNLSTM(n_features=56, window_size=WINDOW),
    "LSTM":        lambda: PureLSTM(n_features=56, window_size=WINDOW),
    "BiLSTM":      lambda: BiLSTM(n_features=56, window_size=WINDOW),
    "Transformer": lambda: TransformerModel(n_features=56, window_size=WINDOW),
}

results = {}
for name, build in MODELS.items():
    print(f"กำลังเทรน {name} ...", end=" ", flush=True)
    results[name] = train_model(build())
    r = results[name]
    print(f"เสร็จ RMSE={r['rmse']:.4f} ({r['epochs']} epochs, {r['seconds']:.0f}s)")

In [ ]:
print(f"\n{'Model':<14}{'RMSE':>9}{'MAE':>9}{'r':>9}{'Params':>11}{'Epochs':>8}{'Time':>8}")
print("=" * 68)
for name in sorted(results, key=lambda n: results[n]["rmse"]):
    r = results[name]
    print(f"{name:<14}{r['rmse']:>9.4f}{r['mae']:>9.4f}{r['r']:>9.4f}"
          f"{r['params']:>11,}{r['epochs']:>8}{r['seconds']:>7.0f}s")

## 6.4 พล็อตเทียบ

In [ ]:
colors = {"CNN-LSTM": "#4f8ef7", "LSTM": "#f97316",
          "BiLSTM": "#a78bfa", "Transformer": "#f472b6"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# กราฟซ้าย: val loss
for name, r in results.items():
    axes[0].plot(r["history"]["val"], label=name, color=colors[name], linewidth=1.8)
axes[0].set_yscale("log")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Val loss (log)")
axes[0].set_title("เส้นไหนลงต่ำและนิ่งเร็ว = ลู่เข้าได้ดี")
axes[0].legend()

# กราฟขวา: RMSE เทียบจำนวนพารามิเตอร์
for name, r in results.items():
    axes[1].scatter(r["params"] / 1000, r["rmse"], s=180,
                    color=colors[name], alpha=0.85, edgecolors="white", linewidth=1.5)
    axes[1].annotate(name, (r["params"] / 1000, r["rmse"]),
                     textcoords="offset points", xytext=(0, 14), ha="center", fontsize=9)
axes[1].set_xlabel("จำนวนพารามิเตอร์ (พัน)")
axes[1].set_ylabel("RMSE (ต่ำ = ดี)")
axes[1].set_title("โมเดลใหญ่กว่า = ดีกว่า จริงหรือ?")

plt.tight_layout()
plt.show()

## 6.5 อ่านผล

ดูกราฟขวา — ถ้า "ใหญ่กว่าดีกว่า" เป็นจริง จุดควรเรียงลงจากซ้ายไปขวา
ลองดูว่ามันเป็นแบบนั้นไหม

**สิ่งที่มักจะเห็นในการทดลองนี้:**

1. **ขนาดไม่ได้ตัดสินผล** — CNN-LSTM มีพารามิเตอร์มากที่สุด (250K)
   แต่ไม่ได้ชนะ บนข้อมูล 675 ตัวอย่างเทรน โมเดลใหญ่มีแนวโน้ม overfit มากกว่าจะได้เปรียบ

2. **Temporal resolution สำคัญ** — CNN-LSTM มี MaxPool 2 ชั้นที่บีบลำดับจาก
   20 เหลือ 5 จุด ข้อมูลรายละเอียดของแนวโน้มหายไปเยอะ
   ส่วน LSTM กับ BiLSTM เก็บครบทั้ง 20 จุด

3. **Transformer ยังไม่ได้เปรียบที่สเกลนี้** — self-attention ต้องการข้อมูลเยอะ
   กว่านี้มากจึงจะเหนือกว่า RNN ลำดับยาวแค่ 20 จุดก็สั้นเกินกว่าจะเห็นข้อได้เปรียบ

In [ ]:
# ทุกโมเดลทำนายตัวอย่างเดียวกัน - ดูว่าเห็นตรงกันไหม
targets = results["LSTM"]["targets"]
order = np.argsort(-targets)

plt.figure(figsize=(12, 4.5))
plt.plot(targets[order], label="ค่าจริง", color="#111", linewidth=2.4)
for name, r in results.items():
    plt.plot(r["preds"][order], label=name, color=colors[name], linewidth=1.1, alpha=0.8)
plt.xlabel("ตัวอย่างใน test set (เรียงตามค่าจริง)")
plt.ylabel("Health Index")
plt.title("ทุกโมเดลทำนายบนตัวอย่างชุดเดียวกัน")
plt.legend(ncol=5, fontsize=9)
plt.tight_layout()
plt.show()

สังเกตว่าทุกโมเดลพลาดที่**จุดเดียวกัน** — ตัวอย่างที่ยากสำหรับโมเดลหนึ่ง
ก็ยากสำหรับทุกโมเดล

นี่บอกเราว่าข้อจำกัดอยู่ที่**ข้อมูล**มากกว่าที่สถาปัตยกรรม
การเปลี่ยนโมเดลจึงช่วยได้จำกัด — ถ้าอยากดีขึ้นจริงต้องปรับปรุงข้อมูลหรือ feature

## 🔧 ลองแก้ดู — ทดลองกับการเปรียบเทียบ


1. เปลี่ยน `seed=42` ใน `train_model` เป็นเลขอื่น (เช่น 1, 7, 123) แล้วรันใหม่ทั้งหมด —
   **อันดับเปลี่ยนไหม?** ถ้าเปลี่ยน แปลว่าความต่างที่เห็นอาจมาจากความบังเอิญ
   ไม่ใช่คุณภาพของสถาปัตยกรรม
2. ลองรันหลาย seed แล้วเก็บค่าเฉลี่ย ± ส่วนเบี่ยงเบน — นี่คือวิธีที่ควรทำในงานวิจัยจริง
3. ลองสร้าง CNN-LSTM ที่ไม่มี MaxPool (แก้ `src/model.py` ให้ `pool_size=1`)
   แล้วดูว่าดีขึ้นไหม — ถ้าดีขึ้นแปลว่าสมมติฐานเรื่อง temporal resolution ถูก

## ❓ เช็คความเข้าใจ

**1. ทำไมต้องรีเซ็ต seed ก่อนสร้างโมเดลแต่ละตัว?**

<details>
<summary>ดูเฉลย</summary>

เพราะน้ำหนักเริ่มต้นถูกสุ่ม ถ้าไม่รีเซ็ต แต่ละโมเดลจะเริ่มจากจุดต่างกัน ความต่างของผลอาจมาจากโชคของการสุ่มแทนที่จะเป็นสถาปัตยกรรม การรีเซ็ตทำให้ทุกตัว เริ่มจากสถานะสุ่มเดียวกัน (เท่าที่โครงสร้างจะเอื้อ)

</details>

**2. BiLSTM ทำได้ดีในการทดลองนี้ แต่ทำไมอาจใช้งานจริงไม่ได้?**

<details>
<summary>ดูเฉลย</summary>

เพราะมันอ่านย้อนกลับจากอนาคตมาหาปัจจุบันภายในหน้าต่าง ในการใช้งานจริงแบบ real-time เราไม่มีข้อมูลอนาคต จึงใช้ได้เฉพาะการวิเคราะห์ย้อนหลังจากข้อมูลที่บันทึกไว้แล้ว

</details>

**3. ถ้าทุกโมเดลพลาดที่ตัวอย่างเดียวกัน บอกอะไรเรา?**

<details>
<summary>ดูเฉลย</summary>

บอกว่าข้อจำกัดอยู่ที่ข้อมูลหรือ label ไม่ใช่ที่ความสามารถของโมเดล การไปลองสถาปัตยกรรมใหม่ ๆ ต่อจะได้ผลตอบแทนน้อย ควรกลับไปดูว่า feature ครบไหม หรือ label สะท้อนความจริงหรือเปล่า

</details>

---

## สรุปบทนี้

- การเทียบที่ยุติธรรมต้องคุมทุกตัวแปรยกเว้นสิ่งที่อยากเทียบ
- โมเดลใหญ่กว่าไม่ได้แปลว่าดีกว่า โดยเฉพาะเมื่อข้อมูลมีจำกัด
- การบีบลำดับเวลาทิ้ง (pooling) แลกมาด้วยรายละเอียดของแนวโน้ม
- ถ้าทุกโมเดลพลาดที่เดียวกัน ปัญหาอยู่ที่ข้อมูล ไม่ใช่สถาปัตยกรรม

[← บทที่ 5](05_first_model_lstm.ipynb) &nbsp;·&nbsp; [สารบัญ](README.md) &nbsp;·&nbsp; **[บทที่ 7 — อ่านผลอย่างมีวิจารณญาณ →](07_reading_results_critically.ipynb)**